# SVG2 stages 5–6 by hand: paragraphs → structured objects → relations (NVIDIA NIM or OpenAI)

**Inputs, all already saved, so no GPU model is run again:**
- `data/stage12/sav_000001_first120_tracks.json`: SAM2 trajectories (stages 1–2). Stage 3 is recomputed from it (deterministic).
- `data/stage4/sav_000001_first120_descriptions.json`: DAM's 40 paragraphs.
- the video, for stage 6's frames.

| stage | what | authors' model | here (default) | code |
|---|---|---|---|---|
| 5 | each paragraph → `{object, attributes, relationships, actions}` (text only) | gpt-5-mini (paper: GPT-4.1-nano) | `nvidia/nemotron-3.5-lightning-30b-a3b`, thinking off | `build_structure_prompt` l.845, `stage5_structure` l.1579 |
| 6 | frames + names + boxes → temporal and spatial relations (vision) | gpt-5 | `moonshotai/kimi-k3` | `_build_relationship_user_content` l.1200, `stage6_relationships` l.1273 |

The models come from `notes/stage56/nim_probe.md`:
- **Not usable for stage 6:** llama-3.2-90b-vision (1 image max), gemma-4-31b (4 max).
- **Usable:** kimi-k3 and nemotron-omni (5 frames OK).
- **Lightning:** only works with thinking **off**; with thinking on, its reasoning leaks into the answer.

**What is the authors' code and what is changed.** The prompts, JSON schemas, frame sampling, input format, box computation and final assembly are their functions, called directly. Only the API call is ours, because NIM needs three small changes:
1. `extra_body` to switch thinking off;
2. `max_tokens` instead of `max_completion_tokens`, and no `detail` field on images;
3. pulling the JSON out of replies that contain extra text.

Each change is marked where it happens.

**Run:** no GPU needed; about 10 min with the comparison models switched on.

In [ ]:
import os, sys, json, time, re, html, getpass, subprocess
REPO = "/kaggle/working/svg2-implementation"
if os.path.isdir(REPO):
    os.chdir(REPO)
subprocess.run("python3 -m pip install -q openai", shell=True)
sys.path.insert(0, os.path.join(os.getcwd(), "third_party/SVG2/pipeline"))
import numpy as np, cv2
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from openai import OpenAI
import svg2_pipeline as P
for p in ("data/stage12/sav_000001_first120_tracks.json", "data/stage4/sav_000001_first120_descriptions.json",
          "third_party/SVG2/pipeline/examples/sav_000001.mp4"):
    print(("OK      " if os.path.exists(p) else "MISSING ") + p)

## Settings and API key

The key is read from `$NVIDIA_API_KEY` / `$OPENAI_API_KEY`, or from `/root/.svg2_keys` (written from Kaggle Secrets), or else asked for in a **hidden** box. **Never type a key into a cell.**

In [ ]:
PROVIDER = "nim"                      # "nim" or "openai"
if PROVIDER == "nim":
    BASE_URL, KEY_ENV = "https://integrate.api.nvidia.com/v1", "NVIDIA_API_KEY"
    STAGE5_MODEL   = "nvidia/nemotron-3.5-lightning-30b-a3b"
    STAGE5_SETTINGS = dict(temperature=0.6, top_p=0.95, extra_body={"chat_template_kwargs": {"enable_thinking": False}})  # NIM change 1
    STAGE5_COMPARE = "moonshotai/kimi-k3"          # bigger model on the first COMPARE_N objects; None to skip
    STAGE6_MODEL   = "moonshotai/kimi-k3"
    STAGE6_COMPARE = "nvidia/nemotron-3-nano-omni-30b-a3b-reasoning"   # None to skip
else:
    BASE_URL, KEY_ENV = None, "OPENAI_API_KEY"
    STAGE5_MODEL, STAGE5_SETTINGS, STAGE5_COMPARE = "gpt-5-mini", {}, None      # the authors' choices
    STAGE6_MODEL, STAGE6_COMPARE = "gpt-5", None
COMPARE_N = 10

if os.path.exists("/root/.svg2_keys"):
    for line in open("/root/.svg2_keys"):
        k, _, v = line.strip().partition("=")
        if k and v and not os.environ.get(k):
            os.environ[k] = v
if not os.environ.get(KEY_ENV):
    os.environ[KEY_ENV] = getpass.getpass(f"{KEY_ENV} (hidden): ")
client = OpenAI(base_url=BASE_URL, api_key=os.environ[KEY_ENV], timeout=300)
print("provider:", PROVIDER, "| stage 5:", STAGE5_MODEL, "| stage 6:", STAGE6_MODEL)

cfg = P.PipelineConfig()
OUT, DATA = "notes/stage56", "data/stage56"
os.makedirs(OUT, exist_ok=True); os.makedirs(DATA, exist_ok=True)
LOG = []
def say(*a):
    s = " ".join(str(x) for x in a); print(s); LOG.append(s)

In [ ]:
# ---- the API call: the authors' logic (strict schema → plain JSON mode → retry) + the NIM changes ----
def extract_json(text):
    """NIM change 3: some models add text or ```json fences around the JSON; take the first {...} that parses."""
    if not text:
        return None
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S)
    for m in re.finditer(r"\{", text):
        depth = 0
        for j in range(m.start(), len(text)):
            depth += {"{": 1, "}": -1}.get(text[j], 0)
            if depth == 0:
                try:
                    return json.loads(text[m.start():j + 1])
                except json.JSONDecodeError:
                    break
    return None

def ask(model, messages, settings, schema=None, max_tokens=4096, tries=3):
    """Returns (parsed_json, raw_text, seconds, mode). Like their SceneGraphStructurer / RelationshipExtractor:
    first try a strict JSON schema, then fall back to plain JSON mode, then to no format at all."""
    formats = ([("strict json_schema", {"type": "json_schema", "json_schema": schema})] if schema else []) + \
              [("json_object", {"type": "json_object"}), ("plain", None)]
    last = None
    for attempt in range(tries):
        for name, fmt in formats:
            kw = dict(model=model, messages=messages, **settings)
            kw["max_completion_tokens" if PROVIDER == "openai" else "max_tokens"] = max_tokens   # NIM change 2
            if fmt:
                kw["response_format"] = fmt
            t0 = time.time()
            try:
                r = client.chat.completions.create(**kw)
                raw = r.choices[0].message.content or ""
                parsed = extract_json(raw)
                if parsed is not None:
                    return parsed, raw, time.time() - t0, name
                last = f"{name}: no JSON in reply ({raw[:120]!r})"
            except Exception as e:
                last = f"{name}: {type(e).__name__}: {str(e)[:200]}"
    raise RuntimeError(last)

def box(html_body, colour="#f6f8fa"):
    return f'<pre style="white-space:pre-wrap;font-size:12px;line-height:1.45;background:{colour};border:1px solid #ccc;padding:8px">{html_body}</pre>'

## 1. Load the saved outputs of stages 2–4

In [ ]:
tracks = json.load(open("data/stage12/sav_000001_first120_tracks.json"))
cleaned = P.stage3_cleanup(cfg, tracks)                         # stage 3, deterministic
descs = json.load(open("data/stage4/sav_000001_first120_descriptions.json"))
frames_all, fps, H, W = P.read_video_frames("third_party/SVG2/pipeline/examples/sav_000001.mp4")
frames = frames_all[:cleaned["total_frames"]]; del frames_all
say(f"{len(cleaned['objects'])} objects after stage 3; {len(descs['objects'])} DAM paragraphs; {len(frames)} frames ({W}x{H}, {fps:.0f} fps)")

## 2. Stage 5: the exact prompt

One call **per object**, text only; the model never sees the video here.
- **grey:** system message;
- **black:** the authors' instructions (paper Table 14);
- **purple:** DAM's paragraph, inserted at the end.

In [ ]:
d1 = next(d for d in descs["objects"] if d["object_id"] == 1)
user = P.build_structure_prompt(d1["description"])
pre, _, _ = user.partition('"""')
display(HTML(box(f'<span style="color:#888">SYSTEM: {html.escape(P.STRUCTURE_SYSTEM_PROMPT)}</span>\n\nUSER: '
                 + html.escape(pre) + '"""<span style="background:#e8e0ff;color:#2b1a66">' + html.escape(d1["description"])
                 + '</span>""".')))
print("JSON schema the answer must follow:", json.dumps(P.SCENE_GRAPH_SCHEMA["properties"]))

## 3. Stage 5: run it for all 40 objects

Their `stage5_structure` does the work: for each object it calls `structurer.structure(paragraph)`, then adds boxes from the masks and assembles the scene graph. We pass a small adapter whose `structure` uses our `ask`.

In [ ]:
class NimStructurer:
    """Stands in for their SceneGraphStructurer (same prompt, same schema, same _validate_scene_graph)."""
    def __init__(self, model, settings):
        self.model, self.settings, self.log = model, settings, []
    def structure(self, description):
        msgs = [{"role": "system", "content": P.STRUCTURE_SYSTEM_PROMPT},
                {"role": "user", "content": P.build_structure_prompt(description)}]
        schema = {"name": "scene_graph", "schema": P.SCENE_GRAPH_SCHEMA, "strict": True}
        parsed, raw, dt, mode = ask(self.model, msgs, self.settings, schema, max_tokens=2048)
        self.log.append((dt, mode))
        return P._validate_scene_graph(parsed)

t0 = time.time()
structurer = NimStructurer(STAGE5_MODEL, STAGE5_SETTINGS)
scene = P.stage5_structure(cfg, descs, cleaned, structurer)        # THEIR stage 5
modes = {m for _, m in structurer.log}
say(f"Stage 5 with {STAGE5_MODEL}: {len(scene['objects'])} objects in {time.time() - t0:.0f} s (output format used: {modes})")
rows = "".join(
    f"<tr><td>{o['object_id']}</td><td>{html.escape(o['description'][:110])}…</td>"
    f"<td><b{' style=color:#c00' if 'uncertain' in o['name'].lower() else ''}>{html.escape(o['name'])}</b></td>"
    f"<td>{html.escape(', '.join(o['attributes']))}</td><td>{html.escape(', '.join(o['actions']))}</td></tr>"
    for o in scene["objects"])
display(HTML("<table style='font-size:11px'><tr><th>id</th><th>DAM paragraph (start)</th><th>name</th><th>attributes</th><th>actions</th></tr>"
             + rows + "</table>"))
names = {o["object_id"]: o["name"] for o in scene["objects"]}
say(f"Labels: {names}")
say(f"Floor check: object 0 (the trampoline floor) was labelled {names.get(0)!r}")
n_unc = sum("uncertain" in n.lower() for n in names.values())
say(f"{n_unc} of {len(names)} labels contain '(uncertain)': stage 6 will not see these objects")

### 3b. Does a bigger model give different labels? (first `COMPARE_N` objects)

In [ ]:
if STAGE5_COMPARE:
    big = NimStructurer(STAGE5_COMPARE, dict(temperature=0.6))
    comp = []
    for o in scene["objects"][:COMPARE_N]:
        r = big.structure(o["description"])
        a, b = set(map(str.lower, o["attributes"])), set(map(str.lower, r["Attributes"]))
        comp.append((o["object_id"], o["name"], r["Object"], len(a & b) / max(len(a | b), 1)))
    for oid, n1, n2, j in comp:
        print(f"id {oid:2d}: {STAGE5_MODEL.split('/')[-1]} → {n1!r:28}  {STAGE5_COMPARE.split('/')[-1]} → {n2!r:28}  attribute overlap {j:.0%}")
    same = sum(n1.lower() == n2.lower() for _, n1, n2, _ in comp)
    say(f"Stage 5 model comparison on {len(comp)} objects: same label {same}/{len(comp)}, mean attribute overlap {np.mean([c[3] for c in comp]):.0%}")

## 4. Stage 6: what the vision model receives

Frames at about 1 per second (`_relationship_frame_indices`): for our 5-second clip that's **frames 0, 24, 48, 72, 96**. For each frame the input has:
- the **image**;
- a **text block** listing the objects visible in that frame, with their stage-5 name and box. Objects whose name contains "uncertain" are left out.

Below is frame 0 exactly as sent, made by their `_build_relationship_user_content`.

In [ ]:
rcfg = cfg.relationship
sampled = P._relationship_frame_indices(scene["total_frames"], fps, rcfg.frame_sample_fps, rcfg.max_frames)
objs6 = [{"object_id": o["object_id"], "name": o["name"], "boxes": o["trajectory"]["boxes"]} for o in scene["objects"]]
user_content = P._build_relationship_user_content(rcfg, frames, sampled, objs6, H, W)        # THEIR input builder
if PROVIDER == "nim":                                                                          # NIM change 2: no 'detail' field
    for item in user_content:
        if item["type"] == "image_url":
            item["image_url"].pop("detail", None)
say(f"Stage 6 input: frames {sampled} → {len(user_content)} items (image, text, image, text, …)")
fig, ax = plt.subplots(1, 1, figsize=(4, 7)); ax.imshow(frames[sampled[0]]); ax.axis("off"); ax.set_title(f"item 1: frame {sampled[0]} (image)")
for o in objs6:
    b = o["boxes"].get(str(sampled[0]))
    if b and "uncertain" not in o["name"].lower():
        ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=1, color="yellow"))
        ax.text(b[0], b[1], f"{o['object_id']}", color="yellow", fontsize=7)
plt.show()
display(HTML(box("item 2 (text):\n" + html.escape(user_content[1]["text"]).replace("}, {", "},\n {"))))

## 5. Stage 6: the two calls (temporal and spatial)

**System prompts** (the authors' text, paper pp. 39–41):
- **temporal:** 6 relation types (functional, stateful, motion, social, attentional, event_level), camera = id −1, each relation with time spans;
- **spatial:** purely 3D layout, no "left of" / "right of".

The first lines of each are shown below. Time spans count **sampled frames**: 0–4 here, about seconds.

In [ ]:
for name, sp in (("TEMPORAL", P.TEMPORAL_RELATION_SYS_PROMPT), ("SPATIAL", P.SPATIAL_RELATION_SYS_PROMPT)):
    display(HTML(box(f"<b>{name} system prompt</b> ({len(sp)} characters; first 900):\n" + html.escape(sp[:900]) + " …", "#fff8e6")))

class NimExtractor:
    """Stands in for their RelationshipExtractor: same system prompts, same schemas, one call per kind."""
    def __init__(self, model):
        self.model, self.log = model, {}
    def extract(self, user_content, kind):
        sys_prompt, schema = {"temporal": (P.TEMPORAL_RELATION_SYS_PROMPT, P.TEMPORAL_RELATION_SCHEMA),
                              "spatial": (P.SPATIAL_RELATION_SYS_PROMPT, P.SPATIAL_RELATION_SCHEMA)}[kind]
        msgs = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_content}]
        settings = dict(reasoning_effort=rcfg.reasoning_effort) if PROVIDER == "openai" else dict(temperature=0.6)
        parsed, raw, dt, mode = ask(self.model, msgs, settings, schema, max_tokens=rcfg.max_completion_tokens)
        self.log[kind] = (dt, mode, raw)
        return parsed.get("relationships", [])

def run_stage6(model):
    ex = NimExtractor(model)
    t0 = time.time()
    g = P.stage6_relationships(cfg, scene, frames, fps, ex)                 # THEIR stage 6
    r = g["relationships"]
    say(f"Stage 6 with {model}: {len(r['temporal'])} temporal + {len(r['spatial'])} spatial relations in {time.time() - t0:.0f} s "
        f"(formats: { {k: v[1] for k, v in ex.log.items()} })")
    return g, ex

graph, ex6 = run_stage6(STAGE6_MODEL)

## 6. Check and read the relations

The schema only checks the *shape* of each relation. Here we also check that it makes sense:
- both ids are objects that were in the input (or −1 = camera);
- no relation from an object to itself;
- time spans lie within 0…4.

In [ ]:
valid_ids = {o["object_id"] for o in objs6 if "uncertain" not in o["name"].lower()} | {-1}
nm = lambda i: "camera" if i == -1 else f"{names.get(i, '?')}({i})"
def wellformed(x):
    return isinstance(x, list) and len(x) >= 4 and isinstance(x[0], int) and isinstance(x[2], int) and isinstance(x[3], list)
def check(rel, kind):
    probs = []
    s, pred, o, spans = rel[0], rel[1], rel[2], rel[3]
    if s not in valid_ids: probs.append(f"unknown subject {s}")
    if o not in valid_ids: probs.append(f"unknown object {o}")
    if s == o: probs.append("self-relation")
    for sp in spans if isinstance(spans, list) else []:
        if not (isinstance(sp, list) and len(sp) == 2 and 0 <= sp[0] <= sp[1] <= len(sampled) - 1):
            probs.append(f"bad span {sp}")
    return probs

def show_relations(g, title):
    r = g["relationships"]; bad = 0; lines = []
    for kind in ("temporal", "spatial"):
        for rel in r[kind]:
            if not wellformed(rel):
                bad += 1; lines.append(f"⚠️ {kind[:4]}  malformed: {rel!r}"); continue
            p = check(rel, kind); bad += bool(p)
            typ = f" [{rel[4]}]" if kind == "temporal" and len(rel) > 4 else ""
            lines.append(f"{'⚠️ ' if p else ''}{kind[:4]}  {nm(rel[0])}  —{rel[1]}→  {nm(rel[2])}  spans {rel[3]}{typ}"
                         + (f"   ← {', '.join(p)}" if p else ""))
    display(HTML(box(f"<b>{html.escape(title)}</b>\n" + html.escape("\n".join(lines)))))
    types = {}
    for rel in r["temporal"]:
        if wellformed(rel) and len(rel) > 4: types[str(rel[4])] = types.get(str(rel[4]), 0) + 1
    say(f"{title}: {len(lines)} relations, {bad} fail the checks; temporal types {types}")
    involving_floor = [x for k in ('temporal', 'spatial') for x in r[k] if wellformed(x) and 0 in (x[0], x[2])]
    say(f"{title}: relations that involve object 0 (the floor, labelled {names.get(0)!r}): {involving_floor}")

show_relations(graph, f"Stage 6 · {STAGE6_MODEL}")

In [ ]:
# draw: on each sampled frame, the boxes of objects that take part in a relation active at that frame
def draw(g, tag):
    r = g["relationships"]; rels = [("T", x) for x in r["temporal"] if wellformed(x)] + [("S", x) for x in r["spatial"] if wellformed(x)]
    fig, axes = plt.subplots(1, len(sampled), figsize=(3.6 * len(sampled), 7.5))
    for k, (ax, f) in enumerate(zip(np.atleast_1d(axes), sampled)):
        ax.imshow(frames[f]); ax.axis("off")
        active = [(t, x) for t, x in rels if any(isinstance(sp, list) and len(sp) == 2 and sp[0] <= k <= sp[1] for sp in x[3])]
        ids = {i for _, x in active for i in (x[0], x[2]) if i != -1}
        for o in objs6:
            b = o["boxes"].get(str(f))
            if o["object_id"] in ids and b:
                ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=1.5, color="yellow"))
                ax.text(b[0] + 2, b[1] + 12, f"{o['object_id']} {o['name'][:14]}", color="black", fontsize=6,
                        bbox=dict(fc="yellow", alpha=0.8, pad=1, lw=0))
        txt = "\n".join(f"{t} {nm(x[0])} {x[1]} {nm(x[2])}" for t, x in active[:12]) + (f"\n… +{len(active) - 12}" if len(active) > 12 else "")
        ax.set_title(f"sampled frame {k} (video frame {f})\n{len(active)} active relations", fontsize=8)
        ax.text(0, H + 15, txt, fontsize=5.5, va="top", family="monospace")
    fig.tight_layout(); fig.savefig(f"{OUT}/relations_{tag}.png", dpi=60, bbox_inches="tight"); plt.show()

draw(graph, STAGE6_MODEL.split("/")[-1])

## 7. Second opinion: the same input to another vision model

Same frames, same names, same prompts; only the model changes. How much of the "ground truth" relations depends on which model wrote them?

In [ ]:
def triples(g, kind):
    return {(x[0], str(x[1]).lower().strip(), x[2]) for x in g["relationships"][kind] if wellformed(x)}
def pairs(g, kind):
    return {(x[0], x[2]) for x in g["relationships"][kind] if wellformed(x)}
if STAGE6_COMPARE:
    graph2, ex6b = run_stage6(STAGE6_COMPARE)
    show_relations(graph2, f"Stage 6 · {STAGE6_COMPARE}")
    draw(graph2, STAGE6_COMPARE.split("/")[-1])
    for kind in ("temporal", "spatial"):
        a, b = triples(graph, kind), triples(graph2, kind)
        pa, pb = pairs(graph, kind), pairs(graph2, kind)
        say(f"Model agreement ({kind}): identical (subject, predicate, object) {len(a & b)} of {len(a)} vs {len(b)}; "
            f"same (subject, object) pair {len(pa & pb)} of {len(pa)} vs {len(pb)}")

## 8. Save

- `data/stage56/…`: the final scene graph per stage-6 model (objects with names, attributes, boxes, masks + relations), the same structure as the authors' stage-6 output. **Push it with `git add -f`.**
- `notes/stage56/`: README with the prompts and all numbers, plus the relation pictures.

In [ ]:
name = "sav_000001_first120"
def slim(g):  # drop the RLE masks from the copy kept in notes (they stay in data/)
    return {**g, "objects": [{k: v for k, v in o.items() if k != "trajectory"} | {"boxes": o["trajectory"]["boxes"]} for o in g["objects"]]}
saved = []
for g, model in [(graph, STAGE6_MODEL)] + ([(graph2, STAGE6_COMPARE)] if STAGE6_COMPARE else []):
    tag = model.split("/")[-1]
    meta = {"stage5_model": STAGE5_MODEL, "stage6_model": model, "provider": PROVIDER}
    p = f"{DATA}/{name}_scene_graph_{tag}.json"; json.dump({**g, "models": meta}, open(p, "w")); saved.append(p)
    json.dump({**slim(g), "models": meta}, open(f"{OUT}/scene_graph_{tag}.json", "w"), indent=1)
lines = ["# Stages 5–6 on `sav_000001` (first 120 frames)", "", *LOG, "",
         "## Stage 5 prompt (one object)", "", "```", P.STRUCTURE_SYSTEM_PROMPT, "", user, "```", "",
         "## Stage 6 input, frame 0 (text part)", "", "```", user_content[1]["text"], "```", ""]
lines += [f"![relations]({p})" for p in sorted(os.listdir(OUT)) if p.startswith("relations_")]
open(f"{OUT}/README.md", "w").write("\n".join(lines) + "\n")
print("saved", saved, "and", OUT)
print("then:  git add -f", " ".join(saved), "\n       git add notebooks/stage56_llm_step_by_step.ipynb", OUT,
      "\n       git commit -m 'stages 5-6 run' && git push")

## 9. What to look for

1. **Section 3:** does the floor (id 0) become "person"? How many labels are "(uncertain)", so that stage 6 never sees them?
2. **Section 3b:** do lightning and kimi agree on labels? On attributes?
3. **Section 6:** do relations make sense on the frames? Any about object 0, the "person" that is really the floor? Any between the two shoe ids that sit on the same shoe (ids 21 and 38)?
4. **Section 7:** how many relations do the two vision models share? Little overlap means SVG2's relations depend heavily on the model that wrote them.